In [ ]:
import os
import ast
import json
import pickle
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
from matplotlib.lines import Line2D
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [3]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR    = CONFIGS['filepaths']['splits']
WEIGHTSDIR   = CONFIGS['filepaths']['weights']
MODELSDIR    = CONFIGS['filepaths']['models']
PREDSDIR     = CONFIGS['filepaths']['predictions']
MODELS       = CONFIGS['experiments']
FIELDVARS    = MODELS['sr']['runs']['sr_gauss']['fieldvars']
NNSEEDS      = MODELS['nn']['seeds']
OPTIMIZEDEQS = MODELS['sr']['optimizedeqs']
PODRUNS      = MODELS['pod']['runs']
NNRUNS       = MODELS['nn']['runs']
ORDER        = ['pod_bl','nn_bl','nn_full','nn_nonparam','nn_gauss','sr_lo','sr_bl','sr_med','sr_hi']
SPLIT        = 'test'
NBINS        = 20
MINSAMPLES   = 50
SRFUNCTIONS  = {
    'cube':lambda x:x**3,
    'square':lambda x:x**2,
    'neg':lambda x:-x,
    'sqrt':np.sqrt,
    'exp':np.exp,
    'log':np.log,
    'abs':np.abs,
    'max':np.maximum,
    'min':np.minimum}
COLORS = {}
LABELS = {}
for name,config in {**PODRUNS,**NNRUNS,**OPTIMIZEDEQS}.items():
    COLORS[name] = config['color']
    LABELS[name] = config['description']

In [ ]:
def kernel_integrate(fields,weights,dsig,mask=None):
    w = fields*weights[None,:,:]*dsig[None,None,:]
    if mask is not None: w = w*mask[:,None,:]
    return w.sum(axis=2)

def physical_prediction(output):
    return np.expm1(tpstd*np.maximum(0.0,np.asarray(output,dtype=float)))

def eval_form(form,variables,constants):
    ns = dict(SRFUNCTIONS); ns.update(variables); ns.update(constants)
    return np.asarray(eval(form,{'__builtins__':{}},ns),dtype=float)

def used_predictors(form,candidates):
    names = {n.id for n in ast.walk(ast.parse(form,mode='eval')) if isinstance(n,ast.Name)}
    return [c for c in candidates if c in names]

def r2(obs,pred):
    mask = np.isfinite(obs)&np.isfinite(pred)
    o,p  = obs[mask],pred[mask]
    return 1-np.sum((o-p)**2)/np.sum((o-o.mean())**2)

def models_for(p):
    return [n for n in MODELPREDICTORS if n in MODELPRED and p in MODELPREDICTORS[n]]

def bin_1d(x,z,nbins=NBINS,minsamples=MINSAMPLES,plo=1,phi=99):
    finite = np.isfinite(x)&np.isfinite(z)
    x,z    = x[finite],z[finite]
    edges  = np.unique(np.percentile(x,np.linspace(plo,phi,nbins+1)))
    n      = len(edges)-1
    xi     = np.clip(np.digitize(x,edges)-1,0,n-1)
    counts = np.bincount(xi,minlength=n)
    sums   = np.bincount(xi,weights=z,minlength=n)
    return 0.5*(edges[:-1]+edges[1:]),np.where(counts>=minsamples,sums/counts,np.nan),counts

def bin_2d(x,y,z,nbins=NBINS,minsamples=MINSAMPLES,plo=1,phi=99):
    finite = np.isfinite(x)&np.isfinite(y)&np.isfinite(z)
    x,y,z  = x[finite],y[finite],z[finite]
    xedges = np.linspace(*np.percentile(x,[plo,phi]),nbins+1)
    yedges = np.linspace(*np.percentile(y,[plo,phi]),nbins+1)
    xi     = np.clip(np.digitize(x,xedges)-1,0,nbins-1)
    yi     = np.clip(np.digitize(y,yedges)-1,0,nbins-1)
    idx    = xi*nbins+yi
    counts = np.bincount(idx,minlength=nbins*nbins).reshape(nbins,nbins)
    sums   = np.bincount(idx,weights=z,minlength=nbins*nbins).reshape(nbins,nbins)
    return 0.5*(xedges[:-1]+xedges[1:]),0.5*(yedges[:-1]+yedges[1:]),np.where(counts>=minsamples,sums/counts,np.nan),counts

In [ ]:
with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    stats = json.load(f)
tpmean = float(stats['tp_mean'])
tpstd  = float(stats['tp_std'])

def flatten(da):
    if 'time' in da.dims: return da.transpose('time','lat','lon').values.ravel()
    return np.tile(da.values,(ntime,1,1)).ravel()

with xr.open_dataset(os.path.join(SPLITSDIR,f'norm_{SPLIT}.h5'),engine='h5netcdf') as ds:
    ntime,nlat,nlon = ds.sizes['time'],ds.sizes['lat'],ds.sizes['lon']
    nsig  = ds.sizes.get('sig',1)
    lat   = ds['lat'].values; lon = ds['lon'].values; dsig = ds['dsig'].values
    fields   = np.stack([ds[v].transpose('time','lat','lon','sig').values.reshape(-1,nsig)
                         for v in FIELDVARS],axis=1)
    surfmask = (ds['surfmask'].transpose('time','lat','lon','sig').values.reshape(-1,nsig)
                if 'surfmask' in ds else None)
    blnorm  = flatten(ds['bl']); lfnorm  = flatten(ds['lf'])
    shfnorm = flatten(ds['shf']); lhfnorm = flatten(ds['lhf'])

kernels = []
for seed in NNSEEDS:
    wpath = os.path.join(WEIGHTSDIR,f'nn_gauss_{seed}_weights.nc')
    if os.path.exists(wpath):
        with xr.open_dataset(wpath,engine='h5netcdf') as ds:
            kernels.append(ds['k'].values)
ki = (np.mean([kernel_integrate(fields,k,dsig,surfmask) for k in kernels],axis=0)
      if kernels else fields.mean(axis=2))
rhk,thetaek,thetaestark = ki[:,0],ki[:,1],ki[:,2]

with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    obsflat = ds.tp.transpose('time','lat','lon').values.ravel()
    lfraw   = flatten(ds['lf']); shfraw = flatten(ds['shf']); lhfraw = flatten(ds['lhf'])
    sefraw  = flatten(ds['sef']); bowraw = flatten(ds['bow']); sdoraw = flatten(ds['sdo'])
    sefnorm = flatten(ds['sef']) if 'sef' in ds else shfnorm+lhfnorm
    months  = np.tile(ds.time.dt.month.values[:,None,None]*np.ones((1,nlat,nlon)),1).ravel().astype(int)

latidx = np.tile(np.arange(nlat)[None,:,None],(ntime,1,nlon)).ravel()
lonidx = np.tile(np.arange(nlon)[None,None,:],(ntime,nlat,1)).ravel()
lats   = lat[latidx]; lons = lon[lonidx]

with open(os.path.join(MODELSDIR,'sr','optimized_equations.pkl'),'rb') as f:
    srparams = pickle.load(f)

In [ ]:
VARINFO = {
    'bl':         (blnorm,      None,   r'$\mathit{B_L}$',           r'$\mathrm{m\,s^{-2}}$'),
    'rh':         (rhk,         None,   'RH',                         '%'),
    'thetae':     (thetaek,     None,   r'$\mathit{\theta_e}$',     'K'),
    'thetaestar': (thetaestark, None,   r'$\mathit{\theta_e^*}$',   'K'),
    'lf':         (lfnorm,      lfraw,  'LF',                         '0–1'),
    'shf':        (shfnorm,     shfraw, 'SHF',                        r'$\mathrm{W\,m^{-2}}$'),
    'lhf':        (lhfnorm,     lhfraw, 'LHF',                        r'$\mathrm{W\,m^{-2}}$'),
    'sef':        (sefnorm,     sefraw, 'SEF',                        r'$\mathrm{W\,m^{-2}}$'),
    'bow':        (bowraw,      bowraw, 'Bowen Ratio',                ''),
    'sdo':        (sdoraw,      sdoraw, 'SDO',                        'm'),
}

VARS = {v: VARINFO[v][0] for v in ['bl','rh','thetae','thetaestar','lf','shf','lhf']}

def physical_predictor(p):
    norm,phys,label,unit = VARINFO[p]
    arr = phys if phys is not None else norm*stats[f'{p}_std']+stats[f'{p}_mean']
    return arr[valid],label,unit

MODELPRED       = {}
MODELPREDICTORS = {}
for name,config in OPTIMIZEDEQS.items():
    entry     = srparams.get(name,{})
    form      = entry.get('form',config['form'])
    constants = entry.get('constants',config['init'])
    pnames    = used_predictors(form,VARS.keys())
    raw       = eval_form(form,{p:VARS[p] for p in pnames},constants)
    MODELPRED[name]       = physical_prediction(raw)
    MODELPREDICTORS[name] = pnames

srhiraw = None
if 'sr_hi' in OPTIMIZEDEQS:
    entry     = srparams.get('sr_hi',{})
    form      = entry.get('form',OPTIMIZEDEQS['sr_hi']['form'])
    constants = entry.get('constants',OPTIMIZEDEQS['sr_hi']['init'])
    pnames    = used_predictors(form,VARS.keys())
    srhiraw   = eval_form(form,{p:VARS[p] for p in pnames},constants)

def load_pred(name,split=SPLIT):
    path = os.path.join(PREDSDIR,f'{name}_{split}_predictions.nc')
    if not os.path.exists(path): return None
    with xr.open_dataset(path,engine='h5netcdf') as ds:
        da = ds['tp'].load()
    if 'seed' in da.dims: da = da.mean('seed')
    pred = da.transpose('time','lat','lon').values.ravel()
    return pred if pred.shape[0]==obsflat.shape[0] else None

for name,config in PODRUNS.items():
    pred = load_pred(name)
    if pred is not None:
        MODELPRED[name] = pred; MODELPREDICTORS[name] = [config['inputvar']]
for name,config in NNRUNS.items():
    pred = load_pred(name)
    if pred is not None:
        MODELPRED[name] = pred; MODELPREDICTORS[name] = config['fieldvars']+config.get('localvars',[])

valid = np.isfinite(obsflat)
for arr in VARS.values(): valid &= np.isfinite(arr)

te  = VARS['thetae'][valid]*stats['thetae_std']+stats['thetae_mean']
tes = VARS['thetaestar'][valid]*stats['thetaestar_std']+stats['thetaestar_mean']
xc2,yc2,pdp,_ = bin_2d(te,tes,obsflat[valid],nbins=50,minsamples=5)
Xg,Yg = np.meshgrid(xc2,yc2,indexing='ij')
fin   = np.isfinite(pdp)
xe,ye,we = Xg[fin],Yg[fin],np.clip(pdp[fin],0,None)
xm,ym = np.average(xe,weights=we),np.average(ye,weights=we)
dx,dy = xe-xm,ye-ym
cov   = np.array([[np.average(dx*dx,weights=we),np.average(dx*dy,weights=we)],
                  [np.average(dx*dy,weights=we),np.average(dy*dy,weights=we)]])
evals,evecs = np.linalg.eigh(cov)
rotw  = evecs[:,np.argmax(evals)]
thetaerotfull = np.full(valid.shape,np.nan)
thetaerotfull[valid] = rotw[0]*te+rotw[1]*tes
VARINFO['thetae_rot'] = (thetaerotfull, thetaerotfull, r'$\mathit{\theta_e}$-rot', 'K')

PREDICTORS = [p for p in ['bl','rh','thetae','thetaestar','lf','shf','lhf','sef','bow','sdo']
              if models_for(p)]
print(f'Rotation weights: [{rotw[0]:.3f}, {rotw[1]:.3f}]')
print(f'Valid: {valid.sum():,} | Loaded: {sorted(MODELPRED.keys())}')
for name,pred in MODELPRED.items():
    print(f'  {LABELS.get(name,name):20s}  R²={r2(obsflat[valid],pred[valid]):.3f}')

In [ ]:
def plot1d(predictors):
    pinfo  = [physical_predictor(p) for p in predictors]
    binned = [bin_1d(x,obsflat[valid]) for x,_,_ in pinfo]
    maxcnt = max(c.max() for _,_,c in binned)
    ncols  = min(3,len(predictors)); nrows = -(-len(predictors)//ncols)
    fig,axs = pplt.subplots(nrows=nrows,ncols=ncols,refwidth=1.5,sharex=False,sharey=True)
    axsf    = np.atleast_1d(axs).ravel()
    for i,(ax,p,(x,label,unit),(xc,obsbin,counts)) in enumerate(zip(axsf,predictors,pinfo,binned)):
        dax = ax.twinx()
        dax.bar(xc,counts,width=xc[1]-xc[0],absolute_width=True,edgecolor='none',
                color='gray5',alpha=0.3)
        dax.format(ylim=(0,1.5*maxcnt),yformatter='sci',
                   ylabel='Sample Count' if (i+1)%ncols==0 or i==len(predictors)-1 else '')
        ax.plot(xc,obsbin,color='k',linewidth=2,zorder=5)
        for name in [n for n in ORDER if n in MODELPRED and p in MODELPREDICTORS.get(n,[])]:
            _,predbin,_ = bin_1d(x,MODELPRED[name][valid])
            ax.plot(xc,predbin,color=COLORS[name],linewidth=1.5,zorder=5)
        ax.format(grid=False,xlabel=f'{label} ({unit})')
    for ax in axsf[len(predictors):]: ax.set_visible(False)
    axsf[0].format(ylabel='Total Precipitation (mm)')
    handles  = [Line2D([],[],color='k',linewidth=2,label='Observed')]
    handles += [Line2D([],[],color=COLORS[n],linewidth=1.5,label=LABELS[n])
                for n in ORDER if n in MODELPRED]
    fig.legend(handles,loc='b',ncols=4)
    return fig

fig = plot1d(PREDICTORS)
pplt.show()

In [ ]:
def plot2d(px,py):
    x,xl,xu = physical_predictor(px); y,yl,yu = physical_predictor(py)
    modelsxy = [n for n in ORDER if n in MODELPRED
                and all(p in MODELPREDICTORS.get(n,[]) for p in [px,py])]
    xc,yc,obsbin,obsc = bin_2d(x,y,obsflat[valid])
    panels  = ['obs']+modelsxy
    fig,axs = pplt.subplots(nrows=1,ncols=len(panels),refwidth=1.5,share=True)
    axsf    = np.atleast_1d(axs).ravel()
    for ax,name in zip(axsf,panels):
        if name=='obs':
            zb = obsbin; cmap,vmin,vmax,extend = 'Blues',0,3,'max'
        else:
            _,_,predbin,_ = bin_2d(x,y,MODELPRED[name][valid])
            zb = predbin-obsbin; cmap,vmin,vmax,extend = 'DryWet',-2,2,'both'
        m = ax.pcolormesh(xc,yc,zb.T,cmap=cmap,vmin=vmin,vmax=vmax,levels=21,extend=extend)
        ax.contour(xc,yc,obsc.T,color='red3',linewidth=0.5)
        ax.format(title='Observed' if name=='obs' else LABELS[name],
                  xlabel=f'{xl} ({xu})',grid=False)
    axsf[0].format(ylabel=f'{yl} ({yu})')
    fig.colorbar(m,loc='r',label='Model − Observed (mm)' if modelsxy else 'Total Precipitation (mm)')
    return fig

for px,py in [('thetae','thetaestar'),('rh','thetaestar'),('rh','thetae')]:
    fig = plot2d(px,py); pplt.show()

In [ ]:
plotvars = ['rh','thetae','thetaestar','bl','shf','lhf','lf','sdo']
fig,axs  = pplt.subplots(nrows=2,ncols=4,refwidth=1.5,sharex=False,sharey=True)
axsf     = np.atleast_1d(axs).ravel()
maxcnt   = max(bin_1d(physical_predictor(v)[0],obsflat[valid])[2].max() for v in plotvars)
for ax,v in zip(axsf,plotvars):
    x,label,unit = physical_predictor(v)
    xc,_,counts  = bin_1d(x,obsflat[valid])
    dax = ax.twinx()
    dax.bar(xc,counts,width=xc[1]-xc[0],absolute_width=True,edgecolor='none',color='gray5',alpha=0.3)
    dax.format(ylim=(0,1.5*maxcnt),yformatter='sci',
               ylabel='Sample Count' if ax in [axsf[3],axsf[7]] else '')
    ax.axhline(0,color='gray',linewidth=0.8,linestyle='--')
    for name in [n for n in ORDER if n in MODELPRED]:
        _,residbin,_ = bin_1d(x,obsflat[valid]-MODELPRED[name][valid])
        ax.plot(xc,residbin,color=COLORS[name],linewidth=1.5)
    ax.format(grid=False,xlabel=f'{label} ({unit})',ylabel='Observed − Predicted (mm)')
handles = [Line2D([],[],color=COLORS[n],linewidth=1.5,label=LABELS[n]) for n in ORDER if n in MODELPRED]
fig.legend(handles,loc='b',ncols=4)
pplt.show()

In [ ]:
obs   = obsflat[valid]
hi    = np.percentile(obs[obs>0],99.9)
edges = np.linspace(0,hi,60); xc = 0.5*(edges[:-1]+edges[1:])

fig,axs = pplt.subplots(nrows=1,ncols=2,refwidth=3,refheight=2,share=False)

obshist,_ = np.histogram(obs,bins=edges,density=True)
axs[0].plot(xc,obshist,color='k',linewidth=2,label='Observed',zorder=6)
for name in [n for n in ORDER if n in MODELPRED]:
    predhist,_ = np.histogram(MODELPRED[name][valid],bins=edges,density=True)
    axs[0].plot(xc,predhist,color=COLORS[name],linewidth=1.5,label=LABELS[name])
axs[0].format(grid=False,xlabel='Total Precipitation (mm)',ylabel='Density',title='PDF',yscale='log')
axs[0].legend(loc='ur',ncols=1)

xc2,_,_ = bin_1d(obs,obs,nbins=20,minsamples=100,plo=0,phi=99)
axs[1].axhline(0,color='gray',linewidth=0.8,linestyle='--')
for name in [n for n in ORDER if n in MODELPRED]:
    _,biasbin,_ = bin_1d(obs,MODELPRED[name][valid]-obs,nbins=20,minsamples=100,plo=0,phi=99)
    axs[1].plot(xc2,biasbin,color=COLORS[name],linewidth=1.5,label=LABELS[name])
axs[1].format(grid=False,xlabel='Observed Precipitation (mm)',
              ylabel='Predicted − Observed (mm)',title='Conditional Bias')
axs[1].legend(loc='ll',ncols=1)
pplt.show()

In [ ]:
lhf   = VARINFO['lhf'][1][valid]; bowen = VARINFO['bow'][1][valid]
invb  = np.where(np.abs(lhf)>5.0, 1.0/np.abs(bowen), np.nan)
xc,obsbin,counts = bin_1d(invb,obsflat[valid])
_,fracneg,_      = bin_1d(invb,(bowen<0).astype(float))

xc3,obsbin3,_    = bin_1d(VARINFO['thetae_rot'][1][valid],obsflat[valid])

fig,axs = pplt.subplots(nrows=1,ncols=3,refwidth=2,share=False)
dax0 = axs[0].twinx()
dax0.bar(xc,counts,width=xc[1]-xc[0],absolute_width=True,edgecolor='none',color='gray5',alpha=0.3)
dax0.format(ylim=(0,4*counts.max()),yformatter='sci',ylabel='Sample Count')
axs[0].plot(xc,obsbin,color='k',linewidth=1.5)
axs[0].format(grid=False,xlabel='1/|Bowen|',ylabel='Total Precipitation (mm)',
              title='Precip vs. 1/|Bowen|')

dax1 = axs[1].twinx()
dax1.bar(xc,counts,width=xc[1]-xc[0],absolute_width=True,edgecolor='none',color='gray5',alpha=0.3)
dax1.format(ylim=(0,4*counts.max()),yformatter='sci',ylabel='Sample Count')
axs[1].plot(xc,fracneg,color='C3',linewidth=1.5)
axs[1].format(grid=False,xlabel='1/|Bowen|',ylabel='Fraction Bowen < 0',
              title='Negative-Bowen Frequency')

axs[2].plot(xc3,obsbin3,color='k',linewidth=2,label='Observed')
for name in [n for n in ORDER if n in MODELPRED]:
    _,predbin3,_ = bin_1d(VARINFO['thetae_rot'][1][valid],MODELPRED[name][valid])
    axs[2].plot(xc3,predbin3,color=COLORS[name],linewidth=1.5,label=LABELS[name])
axs[2].format(grid=False,xlabel=r'$\mathit{\theta_e}$-rot (K)',
              ylabel='Total Precipitation (mm)',title=r'Precip vs. $\mathit{\theta_e}$-rot')
axs[2].legend(loc='ll',ncols=1)
pplt.show()

In [ ]:
obs     = obsflat[valid]
namesr2 = [n for n in ['sr_hi','nn_gauss','sr_med','sr_lo','nn_bl'] if n in MODELPRED]
namesm  = [n for n in ['sr_hi','nn_gauss','sr_med','nn_bl'] if n in MODELPRED]

edges  = np.percentile(obs,np.linspace(0,95,16)); xc = 0.5*(edges[:-1]+edges[1:])
idxs   = np.clip(np.digitize(obs,edges)-1,0,14)
climflat = (np.nanmean(obsflat.reshape(ntime,nlat,nlon),axis=0,keepdims=True)
             *np.ones((ntime,1,1))).ravel()[valid]
r2all  = {'Climatology': r2(obs,climflat)}
r2all.update({LABELS[n]: r2(obs,MODELPRED[n][valid]) for n in namesr2})

mons = months[valid]; lfv = VARINFO['lf'][1][valid]
lav  = lats[valid];   lov = lons[valid]
regions = {
    'Ocean':          lfv<0.1,
    'Land':           lfv>0.9,
    r'5\u201315\u00b0N': (lav>=5)&(lav<15),
    r'15\u201325\u00b0N':(lav>=15)&(lav<=25),
    r'60\u201375\u00b0E':(lov>=60)&(lov<75),
    r'75\u201390\u00b0E':(lov>=75)&(lov<=90)}

In [ ]:
fig,axs = pplt.subplots(nrows=1,ncols=2,refwidth=3,refheight=2,share=False)
axs[0].axhline(0,color='gray',linewidth=0.8,linestyle='--')
for name in namesr2:
    r2bins = [r2(obs[idxs==i],MODELPRED[name][valid][idxs==i])
              if (idxs==i).sum()>=200 else np.nan for i in range(15)]
    axs[0].plot(xc,r2bins,color=COLORS[name],linewidth=1.5,label=LABELS[name])
axs[0].format(xlabel='Observed Precipitation (mm)',ylabel=r'$R^2$',
              title=r'Conditional $R^2$',grid=False)
axs[0].legend(loc='ur',ncols=1)

xlbls = list(r2all.keys()); xvals = list(r2all.values())
clrs  = ['gray']+[COLORS[n] for n in namesr2]
axs[1].bar(np.arange(len(xlbls)),xvals,color=clrs,alpha=0.85)
axs[1].format(ylabel=r'$R^2$',xticks=np.arange(len(xlbls)),xticklabels=xlbls,
              xrotation=25,title=r'Overall $R^2$',grid=False,ylim=(0,1))
pplt.show()

fig,axs = pplt.subplots(nrows=1,ncols=2,refwidth=3.5,refheight=2,share=False)
xmon = np.array([6,7,8])
for name in namesm:
    pred = MODELPRED[name][valid]
    axs[0].plot(xmon,[r2(obs[mons==m],pred[mons==m]) for m in [6,7,8]],
                color=COLORS[name],linewidth=1.5,marker='o',markersize=4,label=LABELS[name])
axs[0].format(xlabel='Month',ylabel=r'$R^2$',xticks=[6,7,8],xticklabels=['Jun','Jul','Aug'],
              title=r'$R^2$ by Month',grid=False)
axs[0].legend(loc='ul',ncols=1)

xreg  = np.arange(len(regions)); width = 0.8/len(namesm)
for i,name in enumerate(namesm):
    pred = MODELPRED[name][valid]
    axs[1].bar(xreg+i*width,[r2(obs[mask],pred[mask]) for mask in regions.values()],
               width=width,color=COLORS[name],label=LABELS[name],alpha=0.85)
axs[1].format(ylabel=r'$R^2$',xticks=xreg+0.4,xticklabels=list(regions.keys()),
              xrotation=20,title=r'$R^2$ by Region',grid=False)
axs[1].legend(loc='ur',ncols=1)
pplt.show()

In [ ]:
KINAMES = {'rh':'KI-RH','thetae':r'KI-$\mathit{\theta_e}$',
           'thetaestar':r'KI-$\mathit{\theta_e^*}$'}
featvars = ['rh','thetae','thetaestar','bl','lf','shf','lhf','sef','sdo']
featarrs = {KINAMES.get(v,VARINFO[v][2]): VARINFO[v][0] for v in featvars}
featarrs['Bowen'] = np.where(np.abs(VARINFO['lhf'][1])>0.01,
                             VARINFO['shf'][1]/VARINFO['lhf'][1], np.nan)

if srhiraw is not None:
    srhiphys  = physical_prediction(srhiraw)
    residphys = obsflat - srhiphys
    residlog  = np.log1p(np.maximum(0,obsflat)) - np.log1p(np.maximum(0,srhiphys))

    corrs = {}
    for fn,fa in featarrs.items():
        x,y = fa[valid],residphys[valid]
        m   = np.isfinite(x)
        corrs[fn] = np.corrcoef(x[m],y[m])[0,1]
    ranked = sorted(corrs.items(),key=lambda x:abs(x[1]),reverse=True)
    print('Feature correlation with obs − SR-HI:')
    for fn,rv in ranked: print(f'  {fn:30s}  r={rv:+.4f}')
else:
    print('sr_hi not found; skipping residual analysis')

In [ ]:
if srhiraw is not None:
    fnames = [k for k,_ in ranked]; rvals = [v for _,v in ranked]
    fig,ax = pplt.subplots(refwidth=3.5,refheight=3)
    ax.barh(np.arange(len(fnames)),rvals,
            color=['blue6' if r>0 else 'red3' for r in rvals],linewidth=0)
    ax.format(yticks=np.arange(len(fnames)),yticklabels=fnames,
              xlabel='Pearson r  (feature vs. obs \u2212 SR-HI)',
              title='Feature\u2013Residual Correlation',xlim=(-0.6,0.6),grid=False)
    ax.axvline(0,color='gray',linewidth=0.7)
    pplt.show()

    plotfeats = [(KINAMES.get(v,VARINFO[v][2]), VARINFO[v][0])
                 for v in ['rh','thetae','thetaestar','bl','lf','shf','sef','sdo']]
    fig,axs  = pplt.subplots(nrows=2,ncols=4,refwidth=2.2,refheight=1.8,
                              sharex=False,sharey=True)
    axsf = np.atleast_1d(axs).ravel()
    for ax,(fname,farr) in zip(axsf,plotfeats):
        xc,means = bin_1d(farr[valid],residlog[valid])[:2]
        ax.plot(xc,means,color='k',linewidth=1.5)
        ax.axhline(0,color='gray',linewidth=0.7,linestyle='--')
        ax.format(xlabel=fname,ylabel='log1p residual',grid=False)
    fig.format(suptitle='log(1+obs) \u2212 log(1+SR-HI)  vs. each feature')
    pplt.show()